In [11]:
## Imports from activity 6: apis_blank
import pandas as pd
import numpy as np
import re
import requests
import yaml
import os

## Imports for wikipedia pulls
import pywikibot as pw
import time
pw.config.user_agent = 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu) Pywikibot/11.6.0'

## imports for test visualization, if needed
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import curve_fit

## set font parameters, if needed
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [16]:
## Part 1: creates foundational dataframe for project, scraped from google searches

## List of countries U.S. has played since 2014. Include year, match outcome, match score, match date, match type.
country_list = ["Ghana", "Portugal", "Germany", "Belgium",
                "Wales", "England", "Iran", "Netherlands", 
                "Paraguay", "Australia", "Turkey", "Bosnia_and_Herzegovina", "Belgium"]

year_played = [2014, 2014, 2014, 2014,
               2022, 2022, 2022, 2022, 
               2026, 2026, 2026, 2026, 2026]

match_outcome = ["win", "draw", "loss", "loss",
                 "draw", "draw", "win", "loss", 
                 "win", "win", "loss", "win", "loss"]

match_score = ["2-1", "2-2", "0-1", "1-2",
              "1-1", "0-0", "1-0", "1-3",
              "4-1", "2-0", "2-3", "2-0", "1-4"]

match_date = ["20140614", "20140622", "20140626", "20140701",
             "20221121", "20221125", "20221129", "20221203",
             "20260612", "20260619", "20260625", "20260701", "20260706"]

match_type = ["group", "group", "group", "knockout",
            "group", "group", "group", "knockout",
            "group", "group", "group", "knockout", "knockout"]

## Create simple dataframe for visualization

basic_info = pd.DataFrame({"country": country_list, "year played": year_played, 
                           "match date": match_date, "outcome": match_outcome, "match type": match_type, "score": match_score,})
basic_info

## Now store the data as a csv to call later so I do not have to rerun code.
basic_info.to_csv('./data/basic_info.csv', index=False)

,country,year played,match date,outcome,match type,score
0,Ghana,2014,20140614,win,group,2-1
1,Portugal,2014,20140622,draw,group,2-2
2,Germany,2014,20140626,loss,group,0-1
3,Belgium,2014,20140701,loss,knockout,1-2
4,Wales,2022,20221121,draw,group,1-1
5,England,2022,20221125,draw,group,0-0
6,Iran,2022,20221129,win,group,1-0
7,Netherlands,2022,20221203,loss,knockout,1-3
8,Paraguay,2026,20260612,win,group,4-1
9,Australia,2026,20260619,win,group,2-0


In [17]:
## Part 2: Access popular article data
## Use pywikibot library to extract main Wiki subcategories for each country category. List of countries defined in the dataframe

site = pw.Site("en", "wikipedia")

## Empty list to store data later.

subcat_data = []

## User-defined function that iterates over the country column in basic_info df, gets all data on subcategories for each country.
def get_wiki_subcategories(countries):
    ## Get each category associated with country, perform a recursive loop
    for cat_name in countries:
        
        cat = pw.Category(site, cat_name)

        # Use try-except to get information on pages per subcategory
        for subcat in cat.subcategories():

            try:

                all_info = subcat.categoryinfo

                num_articles = all_info.get('pages', 0)

                cat_title = subcat.title()

                subcat_data.append({
                    "country": cat_name,
                    "subcategory_title": cat_title,
                    "article_count": num_articles
                })
                ## Need time.sleep() because I kept getting 429 errors, making too many requests.
                time.sleep(2)

            except KeyError:

                 print(f"{subcat.title()}: 0 articles (or info unavailable)")

## call function, use to_list() method
get_wiki_subcategories(basic_info["country"])

subcat_df = pd.DataFrame(subcat_data)

subcat_df

,country,subcategory_title,article_count
0,Ghana,Category:Ghana-related lists,8
1,Ghana,Category:Buildings and structures in Ghana,4
2,Ghana,Category:Culture of Ghana,54
3,Ghana,Category:Economy of Ghana,26
4,Ghana,Category:Education in Ghana,38
...,...,...,...
222,Belgium,Category:Belgian people,2
223,Belgium,Category:Politics of Belgium,40
224,Belgium,Category:Society of Belgium,12
225,Belgium,Category:Images of Belgium,1


In [20]:
## Removes stubs from subcategories since it confounds data
subcat_df_final = subcat_df[~subcat_df['subcategory_title'].str.contains('stubs', na=False)].copy()

## Now eliminate duplicates from repeat matches (Belgium)
subcat_df_final = subcat_df_final.drop_duplicates().copy()

## Format the data better by removing "Category:" using the regex skills we learned in class
subcat_df_final["subcategory_title_fixed"] = subcat_df_final["subcategory_title"].str.replace('^Category:', '', regex=True)

## Now sort by top 4 article count by subcategory for each country
subcat_df_final = (
    subcat_df_final.sort_values(by=["country", "article_count"], ascending=[True, False])
    .groupby('country')
    .head(4)
    .reset_index(drop=True)
    .copy()
)

## Now display presentable dataframe
subcat_df_data = subcat_df_final[["country", "subcategory_title_fixed", "subcategory_title", "article_count"]]

subcat_df_data

subcat_df_data.to_csv("./data/subcategory_data.csv", index=False)

,country,subcategory_title_fixed,subcategory_title,article_count
0,Australia,Culture of Australia,Category:Culture of Australia,70
1,Australia,Politics of Australia,Category:Politics of Australia,57
2,Australia,Education in Australia,Category:Education in Australia,53
3,Australia,Government of Australia,Category:Government of Australia,37
4,Belgium,Culture of Belgium,Category:Culture of Belgium,41
5,Belgium,Politics of Belgium,Category:Politics of Belgium,40
6,Belgium,Government of Belgium,Category:Government of Belgium,35
7,Belgium,Economy of Belgium,Category:Economy of Belgium,22
8,Bosnia_and_Herzegovina,Culture of Bosnia and Herzegovina,Category:Culture of Bosnia and Herzegovina,46
9,Bosnia_and_Herzegovina,Bosnia and Herzegovina templates,Category:Bosnia and Herzegovina templates,40


In [21]:
## Define a function that pulls four most popular articles from each country by ranking via pageviews. First, outline dates
start_date_2014 = 20140601
end_date_2014 = 20140730

start_date_2022 = 20221101
end_date_2022 = 20221231

start_date_2026 = 20260601
end_date_2026 = 20260730

## Create dictionary of start and end dates for each country
cup_windows = {
    'Australia': (start_date_2026, end_date_2026),
    'Belgium': (start_date_2014, end_date_2014),  # or handle 2014/2022 separately if kept distinct
    'Belgium': (start_date_2026, end_date_2026),
    'Bosnia_and_Herzegovina': (start_date_2026, end_date_2026),
    'England': (start_date_2022, end_date_2022),
    'Germany': (start_date_2014, end_date_2014),
    'Ghana': (start_date_2014, end_date_2014),
    'Iran': (start_date_2022, end_date_2022),
    'Netherlands': (start_date_2022, end_date_2022),
    'Paraguay': (start_date_2026, end_date_2026),
    'Portugal': (start_date_2014, end_date_2014),
    'Turkey': (start_date_2026, end_date_2026),
    'Wales': (start_date_2022, end_date_2022),
}

In [24]:
## Now pull article data for each subcategory, long run time.
article_rows = []

def articles_for_subcats(subcat_df_final):
    for _, row in subcat_df_final.iterrows():
        country = row['country']
        subcat_title = row['subcategory_title'] ## pywikibot needs this
        subcat = pw.Category(site, subcat_title)

        ## Routinely getting errors from the pull, use the try, except logic to resolve issues
        try:
            for page in subcat.articles():
                article_rows.append({
                    'country': country,
                    'subcategory_title_fixed': row['subcategory_title_fixed'],
                    'article_title': page.title()
                })
            time.sleep(4)  
        except Exception as e:
            print(f"Error pulling articles from {subcat_title}: {e}")

articles_for_subcats(subcat_df_final)
articles_df = pd.DataFrame(article_rows)

https://en.wikipedia.org/w/api.php
The server may be down.
Status code: 429
User agent: /ipykernel_launcher (wikipedia:en) Pywikibot/11.6.0 (-1 (unknown)) Python/3.13.9.final.0 requests/2.32.5

The text message is:
You are making too many requests to the API.
Please follow the best practices at .
If you are unsure how to get the access you need, contact .
request-id: f2ddb36b-c9fc-4533-a2d3-1f0f763579bd

Set gcmlimit = ['250']
Sleeping for 4.9 seconds, 2026-08-18 19:01:55
https://en.wikipedia.org/w/api.php
The server may be down.
Status code: 429
User agent: /ipykernel_launcher (wikipedia:en) Pywikibot/11.6.0 (-1 (unknown)) Python/3.13.9.final.0 requests/2.32.5

The text message is:
You are making too many requests to the API.
Please follow the best practices at .
If you are unsure how to get the access you need, contact .
request-id: 9512e0e8-b938-46ce-9e13-cd1b3b7cf19c

Set gcmlimit = ['250']
Sleeping for 14.0 seconds, 2026-08-18 19:02:47
https://en.wikipedia.org/w/api.php
The server

In [ ]:
## Now define a user-generated function to call daily article views for the cup windows we defined earlier
## The function calls upon the four most popular subcategories.

headers = {
    'User-Agent': 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu)'
}

def get_daily_pageviews(articles_df, cup_windows, headers):
    all_daily_records = []

    ## Define cleaner variables
    for _, row in articles_df.iterrows():
        country = row['country']
        subcat = row['subcategory_title_fixed']
        article = row['article_title'].replace(' ', '_')
        
        if country not in cup_windows:
            print(f"No date range defined for {country}, skipping {article}")
            continue
        start_date, end_date = cup_windows[country]
        
        pv_query = (
            'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
            f'en.wikipedia.org/all-access/all-agents/{article}/daily/{start_date}/{end_date}'
        )
        
        try:
            pv_resp = requests.get(pv_query, headers=headers)
            if pv_resp.status_code == 200:
                items = pv_resp.json().get('items', [])
                for item in items:
                    all_daily_records.append({
                        'country': country,
                        'subcategory_title_fixed': subcat,
                        'article_title': article,
                        'date': item['timestamp'][:8],  # YYYYMMDD
                        'views': item.get('views', 0)
                    })
            else:
                # article had no pageview data in this window (common for 404s)
                all_daily_records.append({
                    'country': country,
                    'subcategory_title_fixed': subcat,
                    'article_title': article,
                    'date': None,
                    'views': 0
                })
        except Exception as e:
            print(f"Error pulling pageviews for {article}: {e}")
        
        time.sleep(0.2)
    
    return pd.DataFrame(all_daily_records)

daily_views_df = get_daily_pageviews(articles_df, cup_windows, headers)
daily_views_df

In [25]:
## Now visualize articles_df and create a user-defined function to extract the most popular article from
## each of the four most popular categories
articles_df

,country,subcategory_title_fixed,article_title
0,Australia,Culture of Australia,Culture of Australia
1,Australia,Culture of Australia,AC/DC
2,Australia,Culture of Australia,Akubra
3,Australia,Culture of Australia,Anzac spirit
4,Australia,Culture of Australia,Australian citizenship affirmation
...,...,...,...
2211,Wales,Culture of Wales,Welsh Morris dance
2212,Wales,Culture of Wales,Welsh stepdance
2213,Wales,Culture of Wales,Welsh studies
2214,Wales,Culture of Wales,Welsh surnames


In [57]:
## Track top pageviews via the highest mean over the window.

## First calculate mean views per article over each cup window.
mean_views = (
    daily_views_df.groupby(['country', 'subcategory_title_fixed', 'article_title'])['views']
    .mean()
    .reset_index()
    .rename(columns={'views': 'mean_views'})
)

## Display df
mean_views

## Now use sort_values() method and .head(1) to find the top article spot
top_article_per_subcat = (
    mean_views.sort_values('mean_views', ascending=False)
    .groupby(['country', 'subcategory_title_fixed'])
    .head(1)
    .reset_index(drop=True)
)

top_article_per_subcat

,country,subcategory_title_fixed,article_title,mean_views
0,Australia,Culture of Australia,AC/DC,0.000000
1,Australia,Culture of Australia,Akubra,124.816667
2,Australia,Culture of Australia,Anzac_spirit,56.283333
3,Australia,Culture of Australia,Australian_Journal_of_Linguistics,4.706897
4,Australia,Culture of Australia,Australian_citizenship_affirmation,6.866667
...,...,...,...,...
2211,Wales,Welsh language,Welsh_toponymy,45.836066
2212,Wales,Welsh language,William_Salesbury,16.754098
2213,Wales,Welsh language,Wlpan,5.016667
2214,Wales,Welsh language,Y_Fro_Gymraeg,27.295082


,country,subcategory_title_fixed,article_title,mean_views
0,England,Culture of England,The_Beatles,16766.098361
1,Netherlands,Culture of the Netherlands,Sinterklaas,4660.983607
2,Iran,Culture of Iran,Persian_language,4656.245902
3,Wales,Welsh language,Welsh_language,3820.704918
4,Australia,Culture of Australia,Bluey_(TV_series),2951.150000
5,Wales,Culture of Wales,Mari_Lwyd,2132.213115
6,England,Politics of England,Counties_of_England,1808.131148
7,Iran,Politics of Iran,Supreme_Leader_of_Iran,1721.081967
8,Belgium,Culture of Belgium,The_Adventures_of_Tintin,1652.333333
9,Iran,Economy of Iran,Economy_of_Iran,1141.163934


In [4]:
## Part 3: Access census data

## Define census api key, please don't take mine!
census_api = '9fdf899b3427f195717633b957799dd41ae17fdf'
census_api_key = os.environ.get('CENSUS_API_KEY', f'{census_api}')

In [9]:
## I don't feel like writing a function to call for specific country codes within the B05006 table. I will pull these manually instead 
## for the 2022 and 2024 sets

countries_2022 = ['Wales', 'England', 'Iran', 'Netherlands']
codes_2022 = ['B05006_009E', 'B05006_010E', 'B05006_061E', 'B05006_018E']

## 2024 is the most recent available census data, reasonable to take given migration patterns on the order of 5-10 years+.
countries_2024 = ['Paraguay', 'Australia', 'Turkey', 'Bosnia_and_Herzegovina', 'Belgium']
codes_2024 = ['B05006_175E', 'B05006_132E', 'B05006_090E', 'B05006_031E', 'B05006_015E']

census_table_2022 = pd.DataFrame({"country": countries_2022, "code": codes_2022})
census_table_2024 = pd.DataFrame({"country": countries_2024, "code": codes_2024})

census_table_2022
census_table_2024


,country,code
0,Wales,B05006_009E
1,England,B05006_010E
2,Iran,B05006_061E
3,Netherlands,B05006_018E


,country,code
0,Paraguay,B05006_175E
1,Australia,B05006_132E
2,Turkey,B05006_090E
3,Bosnia_and_Herzegovina,B05006_031E
4,Belgium,B05006_015E


In [10]:
## Now pull diaspora data. We just want the city with the highest fraction of diaspora during the world cup.
## We must also pull from the city's total population (located in B01003_001E). 
## I will write a user-defined function that is generalizable to my tables


def diasporaFraction(code, year):
    call_link = f'https://api.census.gov/data/{year}/acs/acs5'

    census_all_data = []
    for x in code:
        params = {
            'get': f'NAME,{x},B01003_001E',
            'for': 'place:*',
            'key': census_api_key
        }

        ## Format to json data immediately, I think I understand requests.get() well enough
        formatted_response = requests.get(call_link, params=params).json()

        ## Convert to df
        census_formatted_data = pd.DataFrame(formatted_response[1:], columns=formatted_response[0])

        census_formatted_data[x] = pd.to_numeric(census_formatted_data[x], errors='coerce')
        census_formatted_data['B01003_001E'] = pd.to_numeric(census_formatted_data['B01003_001E'], errors='coerce')
        census_formatted_data['frac_diaspora'] = census_formatted_data[x] / census_formatted_data['B01003_001E']
        ## Sort by fraction of diaspora
        census_formatted_data = census_formatted_data.sort_values('frac_diaspora', ascending=False)
        ## Now filter for large cities, I got a result that said Buttzville, NJ contained the highest fraction
        ## of Dutch people. The population of Buttzville is 103.
        census_formatted_data = census_formatted_data[census_formatted_data['B01003_001E'] > 50000].head(1)
        census_all_data.append(census_formatted_data)

    return census_all_data
    
diasporaFraction(codes_2022, '2022')

diasporaFraction(codes_2024, '2024')

[                                NAME  B05006_009E  B01003_001E state  place  \
 2768  Laguna Niguel city, California        662.0        64259    06  39248   
 
       frac_diaspora  
 2768       0.010302  ,
                                NAME  B05006_010E  B01003_001E state  place  \
 3337  Santa Monica city, California        651.0        92168    06  70000   
 
       frac_diaspora  
 3337       0.007063  ,
                            NAME  B05006_061E  B01003_001E state  place  \
 2593  Glendale city, California      26869.0       194512    06  30000   
 
       frac_diaspora  
 2593       0.138135  ,
                            NAME  B05006_018E  B01003_001E state  place  \
 4779  Horizon West CDP, Florida        623.0        58595    12  32610   
 
       frac_diaspora  
 4779       0.010632  ]

[                      NAME  B05006_175E  B01003_001E state  place  \
 5345  Weston city, Florida        512.0        68837    12  76582   
 
       frac_diaspora  
 5345       0.007438  ,
                                NAME  B05006_132E  B01003_001E state  place  \
 3343  Santa Monica city, California        423.0        91169    06  70000   
 
       frac_diaspora  
 3343        0.00464  ,
                           NAME  B05006_090E  B01003_001E state  place  \
 5405  Alpharetta city, Georgia        709.0        66855    13  01696   
 
       frac_diaspora  
 5405       0.010605  ,
                        NAME  B05006_031E  B01003_001E state  place  \
 19528  Utica city, New York       2564.0        64217    36  76540   
 
        frac_diaspora  
 19528       0.039927  ,
                                NAME  B05006_015E  B01003_001E state  place  \
 4618  Deerfield Beach city, Florida        355.0        88093    12  16725   
 
       frac_diaspora  
 4618        0.00403  ]

In [61]:
## Part 4: Access sentiment data
## Now simply import and display the the twitter datasets from 2014, 2022.

## Tried converting 2014 file from .txt to .csv but pandas read it weird.
## Instead, read the .txt file as a table
path_to_2014_tweets = "./data/FIFA_2014_sentiment_dataset.txt"
path_to_2022_tweets = "./data/fifa_world_cup_2022_tweets.csv"

## load data
tweets_2014 = pd.read_table(path_to_2014_tweets)
tweets_2014.head()
tweets_2014.info()
## Turns out, this dataset is pretty useless, but good to check anyway. I scoured the internet for anything public/free and could not find anything else
## for the 2014 cup. Unfortunate, but plenty of data from 2022.

tweets_2022 = pd.read_csv(path_to_2022_tweets)
tweets_2022.head()
tweets_2022.info()

,476060286336913408,neutral
0,477584843283259392,positive
1,478036760815878144,positive
2,478021085330673664,positive
3,477207573699891200,positive
4,477537690028900354,positive


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30067 entries, 0 to 30066
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   476060286336913408  30067 non-null  object
 1   neutral             30067 non-null  object
dtypes: object(2)
memory usage: 469.9+ KB


,Unnamed: 0,Date Created,Number of Likes,Source of Tweet,Tweet,Sentiment
0,0,2022-11-20 23:59:21+00:00,4,Twitter Web App,What are we drinking today @TucanTribe \n@MadB...,neutral
1,1,2022-11-20 23:59:01+00:00,3,Twitter for iPhone,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,positive
2,2,2022-11-20 23:58:41+00:00,1,Twitter for iPhone,Worth reading while watching #WorldCup2022 htt...,positive
3,3,2022-11-20 23:58:33+00:00,1,Twitter Web App,Golden Maknae shinning bright\n\nhttps://t.co/...,positive
4,4,2022-11-20 23:58:28+00:00,0,Twitter for Android,"If the BBC cares so much about human rights, h...",negative


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22524 entries, 0 to 22523
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       22524 non-null  int64 
 1   Date Created     22524 non-null  object
 2   Number of Likes  22524 non-null  int64 
 3   Source of Tweet  22524 non-null  object
 4   Tweet            22524 non-null  object
 5   Sentiment        22524 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.0+ MB


In [ ]:
## Store all data for cleaning purposes